# 11 — The training loop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/011-training-loop.ipynb)

Training is a loop: predict, measure the loss, find the slope, take a small step, repeat. Here you fit a line to twenty noisy points, watch the loss fall, and then choose a step size large enough to blow the whole thing up.

## 1. The numbers

There are twenty points, each with one input and one target. The model is a line with a weight and a bias, and the loss is the average squared gap between the line and the points.

In [ ]:
import numpy as np                                    # the array library, preinstalled on Colab

rng = np.random.default_rng(0)                        # a fixed seed: the same data every run
x = np.linspace(-2, 2, 20)                            # 20 inputs spread evenly
y = 3 * x + 2 + rng.normal(0, 0.5, 20)                # the true rule, plus a little noise
w, b = 0.0, 0.0                                       # start the model flat and wrong

pred = w * x + b                                      # 20 predictions, all zero for now
err = pred - y                                        # 20 residuals
loss = (err ** 2).mean()                              # mean squared error: one number
print("x (first five):", np.round(x[:5], 2))          # the inputs
print("y (first five):", np.round(y[:5], 2))          # the targets
print(f"start: w={w:.2f} b={b:.2f} loss={loss:.3f}")  # about 16, the price of being flat

**Line by line**

- `import numpy as np` — loads NumPy, the array library Colab already has.
- `np.random.default_rng(0)` — a random generator with a fixed seed.
- `x = np.linspace(-2, 2, 20)` — twenty inputs between -2 and 2.
- `y = 3 * x + 2 + noise` — the hidden rule the model has to find.
- `w, b = 0.0, 0.0` — the starting parameters: a flat line at zero.
- `pred = w * x + b` — a prediction for every point at once.
- `err = pred - y` — the residual for every point.
- `(err ** 2).mean()` — square, then average: the mean squared error.
- `print("x ...")` — the first five inputs, rounded for reading.
- `print("y ...")` — the matching targets.
- `print(f"start: ...")` — the loss before any training.

## 2. Step by step: one loop over the points

The slope of the average loss is the average of the per-point slopes, so it can be built with a loop. One step then nudges both parameters against that slope.

In [ ]:
def gradients(w, b):                                  # loss and both slopes at one setting
    dw, db, total = 0.0, 0.0, 0.0                      # three accumulators
    for xi, yi in zip(x, y):                           # one iteration per point
        e = w * xi + b - yi                            # residual for this point
        total = total + e ** 2                         # add its squared error
        dw = dw + 2 * e * xi                           # add its slope for w
        db = db + 2 * e                                # add its slope for b
    n = len(x)                                         # how many points there are
    return dw / n, db / n, total / n                   # averages: slope of the mean loss

w0, b0 = 0.0, 0.0                                      # the flat starting line
dw0, db0, L0 = gradients(w0, b0)                       # slope and loss at the start
print(f"start:  w={w0:.2f} b={b0:.2f} loss={L0:.3f}")  # baseline
print(f"slopes: dL/dw={dw0:+.3f} dL/db={db0:+.3f}")    # both negative: raising them helps
w1 = w0 - 0.1 * dw0                                    # one step of size 0.1 in w
b1 = b0 - 0.1 * db0                                    # one step of size 0.1 in b
print(f"step 1: w={w1:.2f} b={b1:.2f} loss={gradients(w1, b1)[2]:.3f}")   # lower already

**Line by line**

- `def gradients(w, b)` — returns the loss and its two slopes.
- `dw, db, total = 0.0, 0.0, 0.0` — accumulators for the two gradients and the loss.
- `for xi, yi in zip(x, y)` — visit the points one at a time.
- `e = w * xi + b - yi` — the residual for this point.
- `total = total + e ** 2` — sum of squared errors.
- `dw = dw + 2 * e * xi` — this point's contribution to the w slope.
- `db = db + 2 * e` — this point's contribution to the b slope.
- `n = len(x)` — how many points were summed.
- `return dw / n, db / n, total / n` — divide by the count to get averages.
- `w0, b0 = 0.0, 0.0` — the flat starting line, named separately.
- `dw0, db0, L0 = gradients(w0, b0)` — slope and loss at the start.
- `print(f"start: ...")` — the baseline loss.
- `print(f"slopes: ...")` — both slopes, pointing uphill.
- `w1 = w0 - 0.1 * dw0` — a step of size 0.1 against the w slope.
- `b1 = b0 - 0.1 * db0` — a step of size 0.1 against the b slope.
- `gradients(w1, b1)[2]` — index 2 picks the loss out of the triple.

## 3. Plot: three learning rates

The learning rate is the size of each step. Too small and the loss crawls; about right and it drops fast; too large and it bounces before settling. The same data, the same start, three step sizes.

In [ ]:
lrs = [0.001, 0.05, 0.5]                           # slow, quick, and almost too big
histories = []                                         # one loss list per learning rate

for lr in lrs:                                         # train once per learning rate
    w_t, b_t = 0.0, 0.0                                # every run starts from the same place
    hist = []                                          # the loss after each step
    for step in range(60):                             # sixty gradient-descent steps
        e = w_t * x + b_t - y                          # residuals at the current parameters
        w_t = w_t - lr * 2 * (e * x).mean()            # step down the w slope
        b_t = b_t - lr * 2 * e.mean()                  # step down the b slope
        hist.append(((w_t * x + b_t - y) ** 2).mean()) # record the new loss
    histories.append(hist)                             # keep this run's curve
    print(f"lr={lr:<5} -> w={w_t:.2f} b={b_t:.2f} loss={hist[-1]:.3f}")   # compare the three

import matplotlib.pyplot as plt                        # plotting library, preinstalled on Colab
plt.figure(figsize=(6, 3))                             # wide, short figure
for lr, hist in zip(lrs, histories):                   # one curve per learning rate
    plt.plot(hist, label=f"lr = {lr}")                 # draw that run
plt.yscale("log")                                      # the losses span several orders of magnitude
plt.xlabel("step")                                     # x-axis label
plt.ylabel("loss (log scale)")                         # y-axis label
plt.title("step size decides how fast, and whether, you arrive")   # the takeaway
plt.legend()                                           # name the curves
plt.show()                                             # render the figure

**Line by line**

- `lrs = [0.001, 0.05, 0.5]` — the three step sizes to compare.
- `histories = []` — a list that will hold one curve per rate.
- `for lr in lrs` — repeat the training run three times.
- `w_t, b_t = 0.0, 0.0` — reset the parameters for each run.
- `hist = []` — that run's loss list.
- `for step in range(60)` — the training loop itself.
- `e = ...` — residuals at the current parameters.
- `w_t = w_t - lr * 2 * (e * x).mean()` — the gradient step for the weight.
- `b_t = b_t - lr * 2 * e.mean()` — the same step for the bias.
- `hist.append(...)` — store the loss after each step.
- `histories.append(hist)` — keep the finished curve.
- `print(f"lr=... ")` — the final parameters and loss for this rate.
- `import matplotlib.pyplot as plt` — the plotting library.
- `plt.figure(...)` — one canvas for all three curves.
- `for lr, hist in zip(lrs, histories)` — one curve per learning rate.
- `plt.plot(hist, ...)` — draw that run.
- `plt.yscale("log")` — a log axis, so all three curves stay readable.
- `plt.xlabel("step")` — names the horizontal axis.
- `plt.ylabel(...)` — notes that the axis is logged.
- `plt.title(...)` — the takeaway.
- `plt.legend()` — names each curve by its learning rate.
- `plt.show()` — draws the figure.

## 4. Interactive: choose the step size

Drag the slider to change the learning rate and the curve is recomputed from scratch. The title reports where the run ended, so a curve that is still falling at step 60 is not the same as one that has settled.

In [ ]:
from ipywidgets import interact                        # sliders that rerun a function

def show(lr_value=0.05):                              # lr_value comes from the slider
    w_i, b_i = 0.0, 0.0                               # start flat again
    hist = []                                         # the loss after each step
    for step in range(60):                            # sixty steps at this step size
        e = w_i * x + b_i - y                         # residuals
        w_i = w_i - lr_value * 2 * (e * x).mean()     # weight step
        b_i = b_i - lr_value * 2 * e.mean()           # bias step
        hist.append(((w_i * x + b_i - y) ** 2).mean())   # record the loss
    plt.figure(figsize=(6, 3))                        # fresh figure on every drag
    plt.plot(hist, color="tab:blue")                  # the loss curve
    plt.yscale("log")                                 # same scale as section 3
    plt.xlabel("step")                                # x-axis label
    plt.ylabel("loss (log scale)")                    # y-axis label
    plt.title(f"lr={lr_value:.3f}: {hist[0]:.1f} -> {hist[-1]:.3f}, w={w_i:.2f} b={b_i:.2f}")   # live readout
    plt.show()                                        # draw it

interact(show, lr_value=(0.001, 1.0, 0.001))          # attach the slider

**Line by line**

- `from ipywidgets import interact` — Colab ships the widgets library.
- `def show(lr_value=0.05)` — the function that redraws one frame.
- `w_i, b_i = 0.0, 0.0` — a fresh start on every drag.
- `hist = []` — the loss list for this run.
- `for step in range(60)` — the same training loop as section 3.
- `e = ...` — residuals at the current parameters.
- `w_i = w_i - lr_value * ...` — the step now uses the slider's rate.
- `b_i = b_i - lr_value * ...` — the same rate on the bias.
- `hist.append(...)` — the loss after each step.
- `plt.figure(...)` — a new canvas for each frame.
- `plt.plot(hist, ...)` — the loss curve.
- `plt.yscale("log")` — the same scale as section 3.
- `plt.xlabel("step")` — names the horizontal axis.
- `plt.ylabel(...)` — notes that the axis is logged.
- `plt.title(f"...")` — first loss, last loss, and the final parameters.
- `plt.show()` — draws the frame.
- `interact(show, lr_value=(...))` — the slider, its range and its step.

## 5. When it breaks: the step that is too big

A step larger than about twice the curvature overshoots the bottom, lands further up the other side, and grows every time. The sign of the loss change flips at every step while the size multiplies, until the numbers pass what a float can hold and become `inf`.

In [ ]:
lr_bad = 1.5                                          # far beyond the stable step size
w_b, b_b = 0.0, 0.0                                    # the same flat start
bad_hist = []                                          # one loss per step

with np.errstate(over="ignore", invalid="ignore"):     # silence the overflow warnings
    for step in range(420):                           # long enough to overflow
        e_b = w_b * x + b_b - y                       # residuals at the current parameters
        w_b = w_b - lr_bad * 2 * (e_b * x).mean()     # the step, now far too large
        b_b = b_b - lr_bad * 2 * e_b.mean()           # same on the bias
        bad_hist.append(((w_b * x + b_b - y) ** 2).mean())   # record the damage

print("first six losses:", [round(v, 1) for v in bad_hist[:6]])   # sign flips, size grows
print(f"loss at step 100: {bad_hist[100]:.2e}")        # already astronomically large
print("non-finite steps:", int(np.sum(~np.isfinite(bad_hist))))   # the tail is inf

plt.figure(figsize=(6, 3))                             # new figure
plt.plot(bad_hist[:40], color="tab:red")               # the first forty steps
plt.yscale("log")                                      # the growth is multiplicative
plt.xlabel("step")                                     # x-axis label
plt.ylabel("loss (log scale)")                         # y-axis label
plt.title("lr too large: oscillation, then overflow to inf")   # the failure
plt.show()                                             # render

**Line by line**

- `lr_bad = 1.5` — a step size well past the stable limit for this data.
- `w_b, b_b = 0.0, 0.0` — the same start as every other run.
- `bad_hist = []` — the loss after each step.
- `np.errstate(over="ignore", invalid="ignore")` — suppresses the overflow warnings.
- `for step in range(420)` — enough steps for the values to overflow.
- `e_b = w_b * x + b_b - y` — residuals, which now have alternating signs.
- `w_b = w_b - lr_bad * ...` — a step so large it lands past the minimum.
- `b_b = b_b - lr_bad * ...` — the same oversized step on the bias.
- `bad_hist.append(...)` — record the loss each time.
- `print("first six losses", ...)` — shows the oscillation before the explosion.
- `print(f"loss at step 100 ...")` — the size after a hundred steps.
- `print("non-finite steps", ...)` — counts the steps that ended as `inf`.
- `plt.figure(...)` — a new canvas.
- `plt.plot(bad_hist[:40], ...)` — only the first forty points fit on a readable chart.
- `plt.yscale("log")` — the growth is multiplicative.
- `plt.xlabel("step")` — names the horizontal axis.
- `plt.ylabel(...)` — notes that the axis is logged.
- `plt.title(...)` — names the failure.
- `plt.show()` — draws the figure.

## Your turn

In section 4, raise the learning rate until the loss at step 60 is worse than at step 0, and note the value where that happens. Then set the data to 200 points instead of 20 and predict whether the largest safe learning rate goes up or down.